<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
# Build keyword artefacts from extracted runs (clean + aligned)
# - Reads app-demo/extracted/{daily,weekly}/**/*.csv
# - Normalizes to one DateTimeIndex for Close & position
# - Computes equity + metrics with explicit (no forward fill) pct_change
# - Writes:
#     AppDemo/artefacts/{ASSET}/metrics_keywords_{D|W}.csv
#     AppDemo/artefacts/{ASSET}/signals_{KW_*}_{D|W}.csv
#     AppDemo/artefacts/{ASSET}/figs/{KW_*}_{D|W}.png
# - Leaves baseline artefacts untouched
# =============================================================================
# Mount Drive (idempotent in Colab)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

# --- project paths on Drive ---------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT = PROJECT_DIR / "app-demo" / "extracted"  # contains daily/ and weekly/
DST_ARTE    = PROJECT_DIR / "AppDemo" / "artefacts"   # Streamlit reads from here

# --- configuration ------------------------------------------------------------
ASSETS   = ["GOLD", "BTC", "OIL", "USDCNY"]
SUBFREQ  = {"daily": "D", "weekly": "W"}  # folder -> freq code
ANNUAL   = {"D": 252, "W": 52}
COST_BPS = 5  # slippage/commission per unit turnover, in basis points

# I want quiet logs from pandas while parsing inconsistent inputs.
pd.options.mode.copy_on_write = True

# --- helpers ------------------------------------------------------------------
def _infer_asset_from_path(p: Path):
    s = str(p).lower()
    if ("gold" in s) or ("gc=f" in s): return "GOLD"
    if "btc" in s:                      return "BTC"
    if ("oil" in s) or ("cl=f" in s):   return "OIL"
    if "usdcny" in s:                   return "USDCNY"
    return None

def _parse_extracted_df(df: pd.DataFrame):
    """
    Normalize an extracted CSV into a tidy DataFrame with a single DateTimeIndex
    and columns: Close, position. If I can't infer required fields, return None.
    """
    # tolerate random casing from various exporters
    lower = {c.lower(): c for c in df.columns}

    date_col  = lower.get("date") or lower.get("timestamp")
    close_col = lower.get("close") or lower.get("price")
    pos_col   = lower.get("position")
    proba_col = lower.get("prob_up") or lower.get("proba") or lower.get("prob")

    if (date_col is None) or (close_col is None):
        return None

    use_cols = [date_col, close_col]
    if pos_col:   use_cols.append(pos_col)
    if proba_col and (proba_col != pos_col): use_cols.append(proba_col)

    df = df[use_cols].copy()

    # enforce datetime, drop bad rows, sort strictly by time
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)

    # build a single DateTimeIndex and align all series onto it
    idx = pd.Index(df[date_col].values)

    close = pd.to_numeric(df[close_col], errors="coerce")
    close = pd.Series(close.values, index=idx).dropna()

    if pos_col:
        pos = pd.to_numeric(df[pos_col], errors="coerce").fillna(0.0).clip(0, 1)
    elif proba_col:
        # default policy: long if prob_up > 0.5
        pos = (pd.to_numeric(df[proba_col], errors="coerce") > 0.5).astype(float)
    else:
        return None

    pos = pd.Series(pos.values, index=idx).reindex(close.index).fillna(0.0)

    out = pd.DataFrame({"Close": close, "position": pos})
    return out

def _equity_and_metrics(df: pd.DataFrame, freq: str):
    """
    Compute strategy equity and summary metrics.
    I explicitly disable pct_change forward fill to avoid misleading returns.
    """
    df = df.sort_index()

    # price returns (no implicit fill)
    ret = df["Close"].pct_change(fill_method=None).fillna(0.0)

    # position is in [0,1]; I consider it "held next bar"
    pos = df["position"].fillna(0.0).clip(0, 1)

    # simple turnover-based cost model
    turnover = pos.diff().abs().fillna(0.0)
    cost = turnover * (COST_BPS / 1e4)

    strat = (pos.shift(1).fillna(0.0) * ret) - cost
    eq = (1.0 + strat).cumprod()

    ann = ANNUAL[freq]
    mu = float(strat.mean() * ann)
    sd = float(strat.std(ddof=0) * math.sqrt(ann))
    sharpe = (mu / sd) if sd > 0 else np.nan
    maxdd = float((eq / eq.cummax() - 1.0).min())

    mets = {
        "Return_Ann": round(mu, 6),
        "Vol_Ann":    round(sd, 6) if np.isfinite(sd) else np.nan,
        "Sharpe":     round(sharpe, 4) if np.isfinite(sharpe) else np.nan,
        "MaxDD":      round(maxdd, 6),
    }
    return eq, mets

def _save_signals(asset: str, strat: str, freq: str, df: pd.DataFrame):
    """
    Persist signals for the app:
    - signal = 1 on entry, 0 on exit (derived from position diff)
    - position (0/1), Close, Date (index), strategy label
    """
    out_dir = DST_ARTE / asset
    (out_dir / "figs").mkdir(parents=True, exist_ok=True)

    sig = df["position"].diff().fillna(0.0).replace({1.0: 1, -1.0: 0}).astype(int)

    payload = pd.DataFrame({
        "Date": df.index,
        "signal": sig,
        "position": df["position"].astype(int),
        "Close": df["Close"],
        "strategy": strat,
    })
    payload.to_csv(out_dir / f"signals_{strat}_{freq}.csv", index=False)

# --- main builder -------------------------------------------------------------
def build_keyword_artefacts():
    # collect rows, then write a single metrics_keywords_{D|W}.csv per asset
    bucket = {(a, f): [] for a in ASSETS for f in ["D", "W"]}

    for subdir, freq in SUBFREQ.items():
        root = SRC_EXTRACT / subdir
        if not root.exists():
            continue

        for csv in root.rglob("*.csv"):
            asset = _infer_asset_from_path(csv)
            if asset is None:
                continue

            try:
                raw = pd.read_csv(csv)
            except Exception:
                continue

            parsed = _parse_extracted_df(raw)
            if parsed is None or parsed.empty:
                continue

            eq, mets = _equity_and_metrics(parsed, freq)

            # strategy label: KW_<prefix of file>
            strat = "KW_" + csv.stem.split("_")[0].upper()

            _save_signals(asset, strat, freq, parsed)

            # equity figure for quick visual validation in-app
            fig_path = DST_ARTE / asset / "figs" / f"{strat}_{freq}.png"
            ax = eq.plot(figsize=(6, 3), title=f"{asset} – {strat} – {freq}")
            ax.grid(True, alpha=0.3)
            ax.get_figure().savefig(fig_path, dpi=140, bbox_inches="tight")
            ax.get_figure().clf()
            plt.close(ax.get_figure())

            bucket[(asset, freq)].append(
                {"asset": asset, "freq": freq, "strategy": strat, **mets}
            )

    # write metrics per asset/freq, dedup by strategy (last-wins)
    for (asset, freq), rows in bucket.items():
        if not rows:
            continue
        out = pd.DataFrame(rows).sort_values("strategy")
        out = out.drop_duplicates("strategy", keep="last").reset_index(drop=True)

        (DST_ARTE / asset).mkdir(parents=True, exist_ok=True)
        out.to_csv(DST_ARTE / asset / f"metrics_keywords_{freq}.csv", index=False)

    # quick console availability table for sanity check
    chk = []
    for a in ASSETS:
        for f in ["D", "W"]:
            base = (DST_ARTE / a / f"metrics_baseline_{f}.csv").exists()
            kw   = (DST_ARTE / a / f"metrics_keywords_{f}.csv").exists()
            sigs = any((DST_ARTE / a).glob(f"signals_*_{f}.csv"))
            figs = any((DST_ARTE / a / "figs").glob(f"*_{f}.png"))
            chk.append({
                "asset": a, "freq": f,
                "metrics_baseline": "✓" if base else "✗",
                "metrics_keywords": "✓" if kw else "✗",
                "any_signals":       "✓" if sigs else "✗",
                "any_figs":          "✓" if figs else "✗",
            })

    print("Artefacts root:", DST_ARTE)
    print(pd.DataFrame(chk).to_string(index=False))

# --- run ----------------------------------------------------------------------
build_keyword_artefacts()
print("✅ Keyword artefacts rebuilt (clean + aligned).")



Mounted at /content/drive
Artefacts root: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
 asset freq metrics_baseline metrics_keywords any_signals any_figs
  GOLD    D                ✓                ✓           ✓        ✓
  GOLD    W                ✓                ✓           ✓        ✓
   BTC    D                ✓                ✓           ✓        ✓
   BTC    W                ✓                ✓           ✓        ✓
   OIL    D                ✓                ✓           ✓        ✓
   OIL    W                ✓                ✓           ✓        ✓
USDCNY    D                ✓                ✓           ✓        ✓
USDCNY    W                ✓                ✓           ✓        ✓
✅ Keyword artefacts rebuilt (clean + aligned).
